# Student Performance: Regression and Classification

This notebook compares Simple Linear Regression, Multiple Linear Regression, and Logistic Regression using a reproducible synthetic dataset of 5,000 students. Regression predicts a continuous exam score; classification predicts whether a student passes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATASET_PATH = "data/student_performance_5000.csv"
df = pd.read_csv(DATASET_PATH)

assert len(df) == 5_000, f"Expected 5,000 rows, got {len(df):,}"
display(df.head())
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
print("Pass rate:", f"{df['passed'].mean():.1%}")

## Simple and Multiple Linear Regression

Both models use the same train/test split. Simple Linear Regression uses only `study_hours`; Multiple Linear Regression uses all four input features. MAE and RMSE are in exam-score points (lower is better), while R-squared measures the fraction of target variance explained (closer to 1 is better).

In [ ]:
FEATURES = ["study_hours", "attendance_pct", "assignment_score", "sleep_hours"]
X = df[FEATURES]
y_score = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_score, test_size=0.2, random_state=42
)

regression_models = {
    "Simple Linear Regression": (LinearRegression(), ["study_hours"]),
    "Multiple Linear Regression": (LinearRegression(), FEATURES),
}
regression_results = []
regression_predictions = {}

for name, (model, feature_names) in regression_models.items():
    model.fit(X_train[feature_names], y_train)
    predictions = model.predict(X_test[feature_names])
    regression_predictions[name] = predictions
    regression_results.append(
        {
            "Model": name,
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R-squared": r2_score(y_test, predictions),
        }
    )

regression_comparison = pd.DataFrame(regression_results).set_index("Model")
display(regression_comparison.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for axis, (name, predictions) in zip(axes, regression_predictions.items()):
    axis.scatter(y_test, predictions, alpha=0.45, s=18)
    axis.plot([0, 100], [0, 100], color="red", linestyle="--")
    axis.set(title=name, xlabel="Actual exam score", ylabel="Predicted exam score")
    axis.set_xlim(0, 100)
    axis.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Logistic Regression and classification metrics

The classifier predicts `passed` from the four input features. A stratified split preserves the pass/fail proportions in both subsets. Standardization is fit only on the training data through a pipeline. Accuracy is the overall fraction correct; precision measures how many predicted passes were actual passes; recall measures how many actual passes were identified; F1 is the harmonic mean of precision and recall.

In [ ]:
y_passed = df["passed"]
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X,
    y_passed,
    test_size=0.2,
    random_state=42,
    stratify=y_passed,
)

logistic_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1_000, random_state=42)
)
logistic_model.fit(X_train_cls, y_train_cls)
y_pred = logistic_model.predict(X_test_cls)

classification_metrics = pd.Series(
    {
        "Accuracy": accuracy_score(y_test_cls, y_pred),
        "Precision": precision_score(y_test_cls, y_pred, zero_division=0),
        "Recall": recall_score(y_test_cls, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test_cls, y_pred, zero_division=0),
    },
    name="Logistic Regression",
)
display(classification_metrics.to_frame().round(3))

matrix = confusion_matrix(y_test_cls, y_pred, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fail", "Pass"],
    yticklabels=["Fail", "Pass"],
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

## Interpretation

Compare regression models using their held-out MAE, RMSE, and R-squared values. The Multiple Linear Regression model can use attendance, assignments, and sleep in addition to study hours. For classification, consider the confusion-matrix error counts alongside precision, recall, and F1; accuracy alone can hide a weaker result for one class.